# Interactive Vision Control — Manual Robot Control with Vision Feedback

**Purpose:** Control the robot manually while seeing real-time detection feedback.

This notebook provides:
- Manual robot control (Forward, Back, Left, Right, Stop)
- Live camera feed with detection overlays
- Vision-assisted control modes (Approach Ball, Return to Basket)
- Real-time motor speed indicators
- Learning mode showing PID calculations

## 1. Setup

In [ ]:
import sys
import os
import time
import yaml
import cv2
import numpy as np
from IPython.display import display, Image
import ipywidgets as widgets
import threading

# Auto-detect project root by searching for config.yaml marker
project_root = os.getcwd()
while not os.path.exists(os.path.join(project_root, 'config.yaml')) and project_root != '/':
    project_root = os.path.dirname(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.hardware.camera import CameraController
from src.perception.ball_detector import BallDetector
from src.perception.obstacle_detector import ObstacleDetector
from src.perception.basket_detector import BasketDetector
from src.control.navigator import Navigator

print("✓ Modules imported")

## 2. Initialize Hardware and Detectors

In [ ]:
# Load config
config_path = os.path.join(project_root, 'config.yaml')
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Initialize camera
camera = CameraController(config)
if not camera.initialize():
    raise Exception("Camera initialization failed!")

camera.center()
camera.look_down()

# Initialize detectors
ball_detector = BallDetector(config)
obstacle_detector = ObstacleDetector(config)
basket_detector = BasketDetector(config)

# Initialize robot (try JetBot, fallback to mock)
try:
    from jetbot import Robot
    robot = Robot()
    print("✓ JetBot robot initialized")
except:
    # Mock robot for testing without hardware
    class MockMotor:
        def __init__(self):
            self.value = 0.0
    
    class MockRobot:
        def __init__(self):
            self.left_motor = MockMotor()
            self.right_motor = MockMotor()
    
    robot = MockRobot()
    print("⚠️  Mock robot initialized (no hardware)")

# Initialize navigator
navigator = Navigator(robot, config)
navigator.set_frame_size(camera.width, camera.height)

print("✓ All systems initialized")

## 3. Interactive Control Interface

In [ ]:
# Create widgets
image_widget = widgets.Image(format='jpeg', width=640, height=480)
status_label = widgets.HTML(value="<b>Status:</b> Ready")
action_label = widgets.HTML(value="<b>Action:</b> None")
motor_label = widgets.HTML(value="<b>Motors:</b> L=0.00, R=0.00")
target_label = widgets.HTML(value="<b>Target:</b> None")

# Speed control
speed_slider = widgets.FloatSlider(
    value=0.15, min=0.05, max=0.30, step=0.05,
    description='Speed:', continuous_update=False
)

# Control state
running = False
auto_mode = None  # None, 'approach_ball', 'return_basket', 'auto_avoid'
manual_command = None
auto_avoid_enabled = False

def update_control_loop():
    global running, auto_mode, manual_command, auto_avoid_enabled
    
    while running:
        frame = camera.read()
        if frame is None:
            time.sleep(0.1)
            continue
        
        # Run detections
        balls = ball_detector.detect(frame)
        obstacle = obstacle_detector.detect_combined(frame)
        basket = basket_detector.detect(frame)
        
        # Draw overlays
        overlay = ball_detector.draw_detections(frame, balls)
        overlay = obstacle_detector.draw_detections(overlay, obstacle)
        overlay = basket_detector.draw_detection(overlay, basket)
        
        # Update display
        _, buffer = cv2.imencode('.jpg', overlay)
        image_widget.value = buffer.tobytes()
        
        # Control logic
        if auto_avoid_enabled and (obstacle['boundary_detected'] or obstacle['obstacle_detected']):
            # Auto-avoidance takes priority
            navigator.avoid_maneuver({'direction': obstacle['turn_direction']})
            action_label.value = f"<b style='color:red'>Action:</b> Auto-Avoiding ({obstacle['turn_direction']})"
        
        elif auto_mode == 'approach_ball' and balls:
            # Approach nearest ball
            target_ball = balls[0]  # Closest
            navigator.approach_target(target_ball)
            color, centroid, distance, _ = target_ball
            action_label.value = f"<b style='color:blue'>Action:</b> Approaching {color} ball"
            target_label.value = f"<b>Target:</b> {color} ball at {distance:.0f}cm"
        
        elif auto_mode == 'return_basket' and basket['basket_found']:
            # Return to basket
            navigator.return_to_basket(basket)
            action_label.value = f"<b style='color:green'>Action:</b> Returning to basket"
            target_label.value = f"<b>Target:</b> Basket at {basket['distance']:.0f}cm, {basket['bearing']:.0f}°"
        
        elif manual_command:
            # Execute manual command
            cmd, speed = manual_command
            if cmd == 'forward':
                navigator.drive_forward(speed)
                action_label.value = f"<b>Action:</b> Forward ({speed:.2f})"
            elif cmd == 'backward':
                navigator.drive_backward(speed)
                action_label.value = f"<b>Action:</b> Backward ({speed:.2f})"
            elif cmd == 'left':
                navigator.turn_left(speed)
                action_label.value = f"<b>Action:</b> Turn Left ({speed:.2f})"
            elif cmd == 'right':
                navigator.turn_right(speed)
                action_label.value = f"<b>Action:</b> Turn Right ({speed:.2f})"
            elif cmd == 'stop':
                navigator.stop()
                action_label.value = "<b>Action:</b> Stopped"
        
        else:
            # No command, stop
            navigator.stop()
            action_label.value = "<b>Action:</b> Idle"
            target_label.value = "<b>Target:</b> None"
        
        # Update motor display
        left_val = robot.left_motor.value
        right_val = robot.right_motor.value
        motor_label.value = f"<b>Motors:</b> L={left_val:+.2f}, R={right_val:+.2f}"
        
        time.sleep(0.05)  # 20 Hz control loop

def start_system(b):
    global running
    if not running:
        running = True
        status_label.value = "<b style='color:green'>Status:</b> Running"
        thread = threading.Thread(target=update_control_loop)
        thread.daemon = True
        thread.start()

def stop_system(b):
    global running, auto_mode, manual_command
    running = False
    auto_mode = None
    manual_command = None
    navigator.stop()
    status_label.value = "<b style='color:red'>Status:</b> Stopped"
    action_label.value = "<b>Action:</b> None"

# Manual control buttons
def move_forward(b):
    global manual_command, auto_mode
    auto_mode = None
    manual_command = ('forward', speed_slider.value)

def move_backward(b):
    global manual_command, auto_mode
    auto_mode = None
    manual_command = ('backward', speed_slider.value)

def turn_left(b):
    global manual_command, auto_mode
    auto_mode = None
    manual_command = ('left', speed_slider.value * 0.7)

def turn_right(b):
    global manual_command, auto_mode
    auto_mode = None
    manual_command = ('right', speed_slider.value * 0.7)

def stop_robot(b):
    global manual_command, auto_mode
    auto_mode = None
    manual_command = ('stop', 0)

# Auto control buttons
def approach_ball(b):
    global auto_mode, manual_command
    manual_command = None
    auto_mode = 'approach_ball'

def return_basket(b):
    global auto_mode, manual_command
    manual_command = None
    auto_mode = 'return_basket'

def toggle_auto_avoid(change):
    global auto_avoid_enabled
    auto_avoid_enabled = change['new']

# Create buttons
start_btn = widgets.Button(description="▶ Start", button_style='success', layout=widgets.Layout(width='100px'))
stop_btn = widgets.Button(description="⏹ Stop", button_style='danger', layout=widgets.Layout(width='100px'))

forward_btn = widgets.Button(description="⬆ Forward", button_style='primary', layout=widgets.Layout(width='120px'))
back_btn = widgets.Button(description="⬇ Back", button_style='primary', layout=widgets.Layout(width='120px'))
left_btn = widgets.Button(description="⬅ Left", button_style='primary', layout=widgets.Layout(width='120px'))
right_btn = widgets.Button(description="➡ Right", button_style='primary', layout=widgets.Layout(width='120px'))
stop_move_btn = widgets.Button(description="⏸ Stop", button_style='warning', layout=widgets.Layout(width='120px'))

approach_btn = widgets.Button(description="🎯 Approach Ball", button_style='info', layout=widgets.Layout(width='150px'))
basket_btn = widgets.Button(description="🗑️ Return to Basket", button_style='info', layout=widgets.Layout(width='150px'))
avoid_toggle = widgets.Checkbox(value=False, description='Auto-Avoid Obstacles')

# Attach handlers
start_btn.on_click(start_system)
stop_btn.on_click(stop_system)
forward_btn.on_click(move_forward)
back_btn.on_click(move_backward)
left_btn.on_click(turn_left)
right_btn.on_click(turn_right)
stop_move_btn.on_click(stop_robot)
approach_btn.on_click(approach_ball)
basket_btn.on_click(return_basket)
avoid_toggle.observe(toggle_auto_avoid, names='value')

# Layout
system_controls = widgets.HBox([start_btn, stop_btn])

manual_controls = widgets.VBox([
    widgets.HTML("<h3>Manual Control</h3>"),
    speed_slider,
    widgets.HBox([widgets.HTML(""), forward_btn, widgets.HTML("")]),
    widgets.HBox([left_btn, stop_move_btn, right_btn]),
    widgets.HBox([widgets.HTML(""), back_btn, widgets.HTML("")])
])

auto_controls = widgets.VBox([
    widgets.HTML("<h3>Vision-Assisted Control</h3>"),
    approach_btn,
    basket_btn,
    avoid_toggle
])

status_panel = widgets.VBox([
    status_label,
    action_label,
    motor_label,
    target_label
])

# Main display
display(widgets.VBox([
    widgets.HTML("<h2>Interactive Vision Control</h2>"),
    system_controls,
    status_panel,
    widgets.HBox([manual_controls, auto_controls]),
    image_widget
]))

## 4. Learning Mode — Understanding PID Control

See how the PID controller calculates motor speeds based on target position.

In [ ]:
learning_output = widgets.Output()

def show_pid_explanation(b):
    with learning_output:
        learning_output.clear_output(wait=True)
        
        frame = camera.read()
        if frame is None:
            print("No frame available")
            return
        
        balls = ball_detector.detect(frame)
        if not balls:
            print("No balls detected - place a ball in view to see PID calculation")
            return
        
        # Get target ball
        color, centroid, distance, area = balls[0]
        cx, cy = centroid
        
        print("=" * 60)
        print("PID CONTROL EXPLANATION")
        print("=" * 60)
        
        print(f"\n📍 Target: {color} ball at {centroid}")
        print(f"   Distance: {distance:.1f} cm")
        
        # Frame center
        frame_center_x = camera.width // 2
        frame_center_y = camera.height // 2
        
        print(f"\n🎯 Frame center: ({frame_center_x}, {frame_center_y})")
        
        # Calculate error
        error_x = cx - frame_center_x
        error_y = cy - frame_center_y
        
        print(f"\n📊 Error:")
        print(f"   X error: {error_x:+d} pixels")
        if error_x > 0:
            print(f"   → Ball is to the RIGHT, need to turn RIGHT")
        elif error_x < 0:
            print(f"   → Ball is to the LEFT, need to turn LEFT")
        else:
            print(f"   → Ball is CENTERED horizontally")
        
        print(f"   Y error: {error_y:+d} pixels")
        if error_y > 0:
            print(f"   → Ball is BELOW center (close)")
        elif error_y < 0:
            print(f"   → Ball is ABOVE center (far)")
        else:
            print(f"   → Ball is CENTERED vertically")
        
        # Simulate PID calculation
        kp = navigator.pid.kp
        
        # Normalized error (-1 to 1)
        norm_error_x = error_x / (camera.width / 2)
        
        print(f"\n🔧 PID Calculation (simplified):")
        print(f"   Kp (proportional gain): {kp}")
        print(f"   Normalized X error: {norm_error_x:+.3f}")
        
        # Simple differential drive
        base_speed = 1.0
        turn_adjustment = norm_error_x * kp
        
        left_speed = base_speed - turn_adjustment
        right_speed = base_speed + turn_adjustment
        
        print(f"\n⚙️  Motor speeds (before scaling):")
        print(f"   Left motor: {left_speed:+.3f}")
        print(f"   Right motor: {right_speed:+.3f}")
        
        if left_speed > right_speed:
            print(f"   → Left faster = turn RIGHT")
        elif right_speed > left_speed:
            print(f"   → Right faster = turn LEFT")
        else:
            print(f"   → Equal speeds = go STRAIGHT")
        
        print(f"\n💡 The robot continuously adjusts motor speeds to center the ball!")

pid_btn = widgets.Button(description="🧠 Explain PID", button_style='primary')
pid_btn.on_click(show_pid_explanation)

display(widgets.VBox([
    widgets.HTML("<h3>Learning Mode</h3>"),
    pid_btn,
    learning_output
]))

## 5. Quick Reference Guide

In [ ]:
guide = widgets.HTML("""
<div style='background-color: #f0f0f0; padding: 15px; border-radius: 5px;'>
<h3>📖 Quick Reference</h3>

<h4>Manual Control:</h4>
<ul>
  <li><b>Forward/Back/Left/Right:</b> Direct motor control at selected speed</li>
  <li><b>Stop:</b> Immediately stop all motors</li>
  <li><b>Speed slider:</b> Adjust movement speed (0.05 to 0.30)</li>
</ul>

<h4>Vision-Assisted Control:</h4>
<ul>
  <li><b>Approach Ball:</b> Uses PID to track and approach nearest detected ball</li>
  <li><b>Return to Basket:</b> Navigates to detected basket using PID</li>
  <li><b>Auto-Avoid:</b> Automatically avoids boundaries and obstacles</li>
</ul>

<h4>Priority System:</h4>
<ol>
  <li>Auto-avoidance (if enabled and obstacle detected)</li>
  <li>Vision-assisted modes (Approach/Return)</li>
  <li>Manual commands</li>
</ol>

<h4>Tips:</h4>
<ul>
  <li>Start with low speed (0.10-0.15) to learn control</li>
  <li>Enable auto-avoid for safety during manual control</li>
  <li>Vision modes work best when target is clearly visible</li>
  <li>Press Stop button for emergency stop</li>
</ul>
</div>
""")

display(guide)

## 6. Cleanup

In [ ]:
# Stop system
running = False
time.sleep(0.5)

# Stop motors
navigator.stop()

# Release camera
camera.release()

print("✓ Cleanup complete")
print("✓ Motors stopped")
print("✓ Camera released")

## Summary

You've learned:
- ✓ Manual robot control with real-time vision feedback
- ✓ Vision-assisted control modes (approach ball, return to basket)
- ✓ How PID control works for tracking targets
- ✓ Priority system for combining manual and automated control
- ✓ Safe operation with auto-avoidance

**Congratulations!** You now understand:
1. Basic camera operations (OpenCV)
2. How the vision system detects objects (HSV, contours, ROIs)
3. Real-time detection performance
4. Vision-based robot control (PID, navigation)

You're ready to work with the robot's vision and control systems!